##### Copyright 2026 Google LLC.

In [2]:
# @title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Gemini API: All about tokens

<a target="_blank" href="https://colab.research.google.com/github/google-gemini/cookbook/blob/main/quickstarts/Counting_Tokens.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" height=30/></a>

An understanding of tokens is central to using the Gemini API. This guide will provide a interactive introduction to what tokens are and how they are used in the Gemini API.

## About tokens

LLMs break up their input and produce their output at a granularity that is smaller than a word, but larger than a single character or code-point.

These **tokens** can be single characters, like `z`, or whole words, like `the`. Long words may be broken up into several tokens. The set of all tokens used by the model is called the vocabulary, and the process of breaking down text into tokens is called tokenization.

For Gemini models, a token is equivalent to about 4 characters. **100 tokens are about 60-80 English words**.

When billing is enabled, the price of a paid request is controlled by the [number of input and output tokens](https://ai.google.dev/pricing), so knowing how to count your tokens is important.


> **Note:** This notebook uses the [Interactions API](https://ai.google.dev/gemini-api/docs/interactions), the latest way to interact with Gemini models. Looking for the `generateContent` version? Check the [archive branch](https://github.com/google-gemini/cookbook/blob/archive/generate-content-api/quickstarts/Counting_Tokens.ipynb).

## Setup

### Install SDK

Install the SDK from [PyPI](https://github.com/googleapis/python-genai).

In [ ]:
%pip install -U -q 'google-genai>=2.0.0'  # 2.0 for Interactions API

### Setup your API key

To run the following cell, your API key must be stored in a Colab Secret named `GEMINI_API_KEY`. If you don't already have an API key, or you're not sure how to create a Colab Secret, see [Authentication](https://github.com/google-gemini/cookbook/blob/main/quickstarts/Authentication.ipynb) for a walkthrough.

In [11]:
from google.colab import userdata

GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')

### Initialize SDK client

With the new SDK you now only need to initialize a client with your API key (or OAuth if using [Vertex AI](https://cloud.google.com/vertex-ai)). The model is now set in each call.

In [13]:
from google import genai

client = genai.Client(api_key=GEMINI_API_KEY)

## Tokens in the Gemini API

### Context windows

The models available through the Gemini API have context windows that are measured in tokens. These define how much input you can provide, and how much output the model can generate, and combined are referred to as the "context window". This information is available directly through [the API](https://ai.google.dev/api/rest/v1/models/get) and in the [models](https://ai.google.dev/models/gemini) documentation.

In this example you can see the `gemini-2.5-flash` model has an 1M tokens context window. If you need more, Pro models have an even bigger 2M tokens context window.

In [16]:
MODEL_ID = "gemini-3-flash-preview"

model_info = client.models.get(model=MODEL_ID)

print("Context window:",model_info.input_token_limit, "tokens")
print("Max output window:",model_info.output_token_limit, "tokens")

Context window: 1048576 tokens
Max output window: 65536 tokens


## Counting tokens

The API provides an endpoint for counting the number of tokens in a request: [`client.models.count_tokens`](https://googleapis.github.io/python-genai/#count-tokens-and-compute-tokens). You pass the same arguments as you would to [`client.models.interactions.create`](https://googleapis.github.io/python-genai/#generate-content) and the service will return the number of tokens in that request.

Select the model you want to use in this guide:

In [19]:
MODEL_ID = "gemini-3-flash-preview" # @param ["gemini-3.1-flash-lite", "gemini-3-flash-preview", "gemini-3.1-pro-preview"] {"allow-input":true, isTemplate: true}

### Text tokens

In [21]:
response = client.models.count_tokens(
    model=MODEL_ID,
    contents="What's the highest mountain in Africa?",
)
print("Prompt tokens:",response.total_tokens)

Prompt tokens: 10


When you call `client.interactions.create` the response object has a `usage` field that includes the token count information.

In [23]:
interaction = client.interactions.create(
    model=MODEL_ID,
    input="The quick brown fox jumps over the lazy dog."
)
print(interaction.steps[-1].content[0].text)

That is a classic **pangram**—a sentence that contains every letter of the English alphabet at least once.

Because it is relatively short and makes sense, it has been used for over a century to test typewriters, computer keyboards, fonts, and other displays where the user needs to see how every letter looks.

Would you like to see some other examples of pangrams? 

*   **"Pack my box with five dozen liquor jugs."** (This one is even shorter!)
*   **"Sphinx of black quartz, judge my vow."** (Often considered the "coolest" sounding one.)
*   **"How quickly daft jumping zebras vex."**
*   **"Bright vixens jump; dozy fowl quack."**


In [24]:
# Note: The exact field names may differ in Interactions API
# Check interaction.usage for token information
print(interaction.usage)

Usage(cached_tokens_by_modality=None, grounding_tool_count=None, input_tokens_by_modality=[InputTokensByModality(modality='text', tokens=11)], output_tokens_by_modality=None, tool_use_tokens_by_modality=None, total_cached_tokens=0, total_input_tokens=11, total_output_tokens=160, total_thought_tokens=382, total_tokens=553, total_tool_use_tokens=0)


In case you are using [context caching](https://ai.google.dev/gemini-api/docs/caching), the number of cached token will be indicated in `response.usage_metadata.cached_content_token_count`.

### Multi-modal tokens

All input to the API is tokenized, including images or other non-text modalities.

Images are considered to be a fixed size, so they consume a fixed number of tokens, regardless of their display or file size.

Video and audio files are converted to tokens at a fixed per second rate.

The current rates and token sizes can be found on the [documentation](https://ai.google.dev/gemini-api/docs/tokens?lang=python#multimodal-tokens)

#### Inline content

You can pass media directly by URL:

In [ ]:
from google.genai import types

response = client.models.count_tokens(
    model=MODEL_ID,
    contents=[
        types.Part(
            file_data=types.FileData(
                file_uri="https://storage.googleapis.com/generativeai-downloads/images/organ.jpg",
                mime_type="image/jpeg",
            )
        )
    ],
)

print("Prompt with image tokens:", response.total_tokens)

You can try with different images and should always get the same number of tokens, that is independent of their display or file size. Note that an extra token seems to be added, representing the empty prompt.

#### Files API

The model sees identical tokens if you upload parts of the prompt through the files API instead:

In [ ]:
from google.genai import types\n\nresponse = client.models.count_tokens(\n    model=MODEL_ID,\n    contents=types.Part.from_uri(\n        file_uri="https://storage.googleapis.com/generativeai-downloads/images/organ.jpg",\n        mime_type="image/jpeg",\n    ),\n)\n\nprint("Prompt with image tokens:", response.total_tokens)

Audio and video are each converted to tokens at a fixed rate of tokens per minute.

In [ ]:
import subprocess\nfrom google.genai import types\n\nurl = "https://storage.googleapis.com/generativeai-downloads/data/State_of_the_Union_Address_30_January_1961.mp3"\n\n# Compute duration programmatically\ncmd = [\n    "ffprobe",\n    "-v",\n    "error",\n    "-show_entries",\n    "format=duration",\n    "-of",\n    "default=noprint_wrappers=1:nokey=1",\n    url,\n]\nduration = float(subprocess.check_output(cmd).decode().strip())\n\nresponse = client.models.count_tokens(\n    model=MODEL_ID,\n    contents=types.Part.from_uri(\n        file_uri=url,\n        mime_type="audio/mp3",\n    ),\n)\n\nprint("Prompt with audio tokens:", response.total_tokens)\nprint("Tokens per second:", response.total_tokens / duration)

As you can see this corresponds to about 32 tokens per second of audio.

### Chat, tools and caching

Chat, tools and caching are currently not supported by the unified SDK `count_tokens` method. This notebook will be updated when that will be the case.

In the meantime you can still check the token used after the call using the `usage_metadata` from the response. Check the [context caching documentation](https://ai.google.dev/gemini-api/docs/caching) for an example.

## Further reading

For more on token counting, check out the [documentation](https://ai.google.dev/gemini-api/docs/tokens?lang=python#multimodal-tokens) or the API reference:

* [`countTokens`](https://ai.google.dev/api/rest/v1/models/countTokens) REST API reference,
* [`count_tokens`](https://googleapis.github.io/python-genai/#count-tokens-and-compute-tokens) Python API reference,